In [1]:
# basic configuration 
import logging
from typing import Any , Optional
from pydantic import BaseModel, AnyUrl, EmailStr, ValidationError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('ScrapTracer')

class ScraperTargetError(Exception):
    def __init__(self, message: str , error: dict[str, Any]):
        super().__init__(message)
        self.error = error


class ScraperTarget(BaseModel):
    target_url: AnyUrl
    admin_email: EmailStr
    max_pages: int|None = 10
    keywords: str | list[str]

def process_scraper_configs(raw_dict: dict[str, Any]) -> ScraperTarget:
    try:
        return ScraperTarget(**raw_dict)
    except ValidationError as err:
        error_summary = []
        for e in err.errors():
            field_path = "->".join(str(loc) for loc in e['loc'])
            error_summary.append(f"[{field_path}] {e['msg']} (Got: {e['input']})")

        raise ScraperTargetError(
            message= "Payload failure accured.",
            error = error_summary
        ) from err


In [2]:
# testing phase of above implemented code: 
if __name__ == "__main__":
        # Test Payload 1: Fully Valid (Keywords as a single string, max_pages as int)
    valid_payload_single = {
        "target_url": "https://news.ycombinator.com",
        "admin_email": "crawler@ai-data.com",
        "max_pages": 5,
        "keywords": "artificial intelligence"
    }
    # Test Payload 3: Invalid (Invalid email format -> Should raise ScraperConfigError)
    invalid_payload = {
        "target_url": "https://scraping-service.com",
        "admin_email": "majid.12@gcom",  # Invalid Email!
        "max_pages": 20,
        "keywords": "data science"
    }

    
    print("......Testing Valided Payload.....")
    try:
        process_scraper_configs(valid_payload_single)
        print(f"Successfully scraped data payload: {valid_payload_single}")
    except ScraperTargetError as err:
        for e in err.error:
            print(e)
    

......Testing Valided Payload.....
Successfully scraped data payload: {'target_url': 'https://news.ycombinator.com', 'admin_email': 'crawler@ai-data.com', 'max_pages': 5, 'keywords': 'artificial intelligence'}


In [2]:
# testing phase of above implemented code: 
if __name__ == "__main__":
    # Test Payload 3: Invalid (Invalid email format -> Should raise ScraperConfigError)
    invalid_payload = {
        "target_url": "http://scraping-servicecom",
        "admin_email": "majid.12@gcom",  # Invalid Email!
        "max_pages": 20,
        "keywords": "data science"
    }

    
    print("......Testing Invalid Payload.....")
    try:
        process_scraper_configs(invalid_payload)
        print(f"Successfully scraped data payload: {invalid_payload}")
    except ScraperTargetError as err:
        for e in err.error:
            print(e)
    

......Testing Invalid Payload.....
[admin_email] value is not a valid email address: The part after the @-sign is not valid. It should have a period. (Got: majid.12@gcom)
